# Module 1: Your first agent, from scratch and with a framework

In this module you will call a self-hosted model with a raw HTTP request. After that you will use the [ReAct pattern](https://arxiv.org/abs/2210.03629) to build the agent loop **by hand** so you understand exactly what an agent is, then do the same thing with the [Strands](https://strandsagents.com) framework in a fraction of the code. The running example is an **Akamai Cloud Solutions Architect** assistant.

## Learning objectives
- Call a self-hosted model directly with a raw request to the vLLM API
- Read `finish_reason` and drive a tool-calling loop on it
- Build the agent loop by hand with native tool calling, so a framework is never magic
- Do the same thing with Strands, with far less code
- Add built-in tools and your own tools with the `@tool` decorator

## Prerequisites
- Python 3.11+
- A model endpoint: vLLM on Akamai Cloud, or any OpenAI-compatible endpoint
- About 20 minutes

References: [Strands Agents](https://strandsagents.com) · [ReAct paper](https://arxiv.org/abs/2210.03629) · [OpenAI chat API](https://platform.openai.com/docs/api-reference/chat) · [vLLM tool calling](https://docs.vllm.ai/en/latest/features/tool_calling.html) · [Akamai Cloud GPUs](https://www.linode.com/products/gpu/)

## Agent design basics

An agent is a loop. The model decides what to do. If it wants a tool, your code runs
the tool and hands back the result, and the model decides again. When it does not want
a tool, it answers. The model is the brain. The tools are the hands.

![The agent loop: you ask, the model and a tool round-trip until the model is done, then it answers](../images/01_agent_architecture.png)

## 1. Setup

Install `httpx` to call the model using plain HTTP. We also install strands agents and strands agents tools to use later in this lab.

In [ ]:
%pip install -q httpx python-dotenv "strands-agents[openai]" strands-agents-tools

## 2. Configure url, model, and api key

These values are loaded from your `.env` (copy `.env.example` first). The defaults assume a
self-hosted vLLM endpoint serving Qwen2.5. Any OpenAI-compatible endpoint works.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads the repo .env if present

BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
MODEL_ID = os.getenv("VLLM_MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")
API_KEY = os.getenv("VLLM_API_KEY", "placeholder")

print(f"Model: {MODEL_ID}")
print(f"Endpoint: {BASE_URL}")

## 3. Send your first model request

An LLM call is just an HTTP POST. Here we are not using an SDK nor an agent framework. We are only using the exposed vLLM API that is serving our model inference. When you send a `messages` array, you get back a model response that contains a `choices` array. Inside of the choices array you will see a key of `finish_reason` that provides you with *why* the model stopped its response.

Here are some of the common reasons:
- `stop` if the model hit a natural stop point or a provided stop sequence
- `length` if the maximum number of tokens specified in the request was reached
- `content_filter` if content was omitted due to a flag from our content filters
- `tool_calls` if the model called a tool, 
- `function_call` (deprecated) if the model called a function

In [ ]:
import httpx, json

def chat(messages, tools=None, temperature=0.2):
    """POST to the vLLM (OpenAI-compatible) chat endpoint and return the raw JSON."""
    body = {"model": MODEL_ID, "messages": messages, "temperature": temperature}
    if tools:
        body["tools"] = tools
        body["tool_choice"] = "auto"
    resp = httpx.post(
        f"{BASE_URL}/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json=body,
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()

SYSTEM = "You are an Akamai Cloud Solutions Architect. Be concise and developer to developer."

data = chat([
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "In one sentence, what is Akamai Cloud good for?"},
])

print(f"Full model response: {data}")

choice = data["choices"][0]
print(f"finish_reason: {choice['finish_reason']}")
print(f"Agent response: {choice['message']['content']}")

## 4. An LLM is not an agent

A large language model on its own depends on the real world. If you ask it what is the weather today in Boston? The model will not know what today is nor have the ability to check the weather. The model can only answer based on what was in its training data and/or in the context you provide it. And the training of the model has a cutoff date. This date is when the model is frozen in time. By default it has no clock and cannot run a calculation. The `finish_reason` at this point is still `stop`, but the answer is a guess.

In [ ]:
data = chat([
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "What is today's date?"},
])
print(data["choices"][0]["message"]["content"])

## 5. Build the agent loop by hand

To act on the real world the model needs tools and a loop. The modern form of the
[ReAct](https://arxiv.org/abs/2210.03629) pattern uses *native tool calling*: you send
the model a list of tool schemas, and when it wants one, the API returns
`finish_reason == "tool_calls"` with the call. Your code runs it, appends the result,
and loops. When the model is done it returns `finish_reason == "stop"`.

Below we create our first tools. They are plain Python functions, plus a JSON schema that tells the model how to call each one.

In [ ]:
import re
from datetime import datetime, timezone

def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression like '0.108 * 730 * 3'."""
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression.strip()):
        return "error: only arithmetic is allowed"
    try:
        return str(eval(expression))  # safe: the regex above allows only arithmetic
    except Exception as exc:
        return f"error: {exc}"

def current_time() -> str:
    """Return the current UTC time."""
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

PYTOOLS = {"calculator": calculator, "current_time": current_time}

# This is the schema that will be provided to the model to tell it what tools are available
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate an arithmetic expression, for example 0.108 * 730 * 3.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "current_time",
            "description": "Get the current UTC date and time.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
]

Now create and run the loop. It branches on `finish_reason`. This is the ReAct loop. It showcases what an agent is.

In [ ]:
def run_agent(question, max_steps=6):
    messages = [
        {"role": "system", "content": SYSTEM + " Use the calculator for any arithmetic."},
        {"role": "user", "content": question},
    ]
    # Runs the loop for a maximum of max_steps
    for step in range(1, max_steps + 1):
        # Get the model response
        data = chat(messages, tools=TOOLS_SCHEMA)  # --> (Reason)
        choice = data["choices"][0]
        reason = choice["finish_reason"]
        message = choice["message"]
        print(f"step {step}: finish_reason = {reason}")

        if reason == "tool_calls":
            messages.append(message)  # the assistant turn that asked for the tools
            # Loop through the tool calls and run them
            for call in message["tool_calls"]:
                # Get the tool name
                name = call["function"]["name"]
                print(f"Agent is calling tool: {name}")
                # Get the tool arguments
                args = json.loads(call["function"]["arguments"] or "{}")
                # Looks up the tool in the PYTOOLS dictionary and runs it with the arguments
                result = PYTOOLS[name](**args)   # --> (Action)
                print(f"   -> {name}({args}) = {result}")
                # Append the tool result to the messages
                messages.append({"role": "tool", "tool_call_id": call["id"], "content": str(result)})  # --> (Observation)
            
            # Send the tool results back to the model
            continue  # --> Reason again (now over the observation)

        # finish_reason == "stop": the model is done
        return message["content"]  # --> (done)
    return "stopped: reached max steps"

# Run the agent
answer = run_agent(
    "A GPU plan costs $0.108 per hour. What is the monthly cost of running 3 of them? "
    "A month is 730 hours."
)
print(f"\nFINAL ANSWER: {answer}")

That is an agent: a loop that branches on `finish_reason`, runs tools, observes the results, and stops when the model is done. You the developer did not have to parse any free text. The API told you what the model wanted.

Notice the plumbing you wrote: tool schemas, the message list, appending tool results with the right `tool_call_id`, the loop and its limit. The need to write this plumbing is what an agent framework removes.

## 6. The same agent, with an agent framework

In this example we will showcase how an agent framework like Strands Agents runs that loop for you. It also generates the tool schemas from your function signatures and docstrings, so you write tools, not plumbing. 

Below we will code the same capability with the Strands Agents built-in `current_time` and `calculator` tools.

In [ ]:
from strands import Agent
from strands.models.openai import OpenAIModel
from strands_tools import current_time, calculator

model = OpenAIModel(
    client_args={"api_key": API_KEY, "base_url": BASE_URL},
    model_id=MODEL_ID,
    params={"temperature": 0.2},
)

agent = Agent(
    model=model,
    tools=[current_time, calculator],
    system_prompt=SYSTEM + " Use the calculator for all arithmetic.",
)

response = agent(
    "What is today's date, and the monthly cost of running 3 GPUs at $0.108 per hour "
    "for a 730 hour month?"
)

print("\n")
# Print the AgentResult object
print(repr(response))

Notice there was no need to write a tool json schema by hand, no message list, no `finish_reason` branch, no loop. 

Strands ran the same loop you wrote and handled the tool plumbing. That is what a framework provides you, and why the rest of this workshop will use Strands Agents.

>NOTE: Strands uses a `callback_handler` that, by default, prints streaming text output and tool invocations to stdout.

## 7. Building your own tools

Built-in tools are useful for common repeatable use cases. When you build agents in production your agents will need to call your code. The `@tool` decorator in Strands turns a function into a tool, and the docstring is the contract the model reads to decide when to call it. Strands builds the JSON schema from the signature for you.

In [ ]:
from strands import tool

@tool
def get_akamai_region_id(city: str) -> str:
    """Return the Akamai Cloud region id for a city name.

    Use this to convert a human city (for example 'Chicago') into the region id the
    Akamai Cloud API expects (for example 'us-ord').
    Args:
        city: The human-readable city name.
    Returns:
        The Akamai Cloud region id for the city.
    """
    table = {"chicago": "us-ord", "seattle": "us-sea", "osaka": "jp-osa", "amsterdam": "nl-ams"}
    return table.get(city.strip().lower(), f"unknown city: {city}")

# Instantiate the agent
agent = Agent(
    model=model,
    tools=[current_time, calculator, get_akamai_region_id],
    system_prompt=SYSTEM + " Use the calculator for all arithmetic.",
)

# Invoke the agent
response = agent("Which Akamai Cloud region should I use for Chicago?")

## 8. Two vLLM configurations to know

Self-hosting models requires an understanding of the hosted APIs that provides them. When you serve vLLM there are a few things to know:

1. **Tool calling must be enabled on the server.** vLLM sets `finish_reason` to
   `tool_calls` only when started with the right flags. For Qwen2.5:

   ```bash
   vllm serve Qwen/Qwen2.5-7B-Instruct \
     --enable-auto-tool-choice \
     --tool-call-parser hermes
   ```

   Without them, the model writes the tool call as plain text, `finish_reason` stays
   `stop`, and your loop never fires a tool. 

   Above we are showing the CLI approach but if using terraform or kubernetes you would configure this in your manifest file. 

2. **Some vLLM builds reject an empty tools array.** That is why every Strands agent
   here carries at least one tool in all examples. When you truly want no tools, send a request with no
   `tools` field at all, as in section 3.

The `workshop/scripts/verify_env.py` check probes both. 

Reference: [vLLM tool calling](https://docs.vllm.ai/en/latest/features/tool_calling.html).

## Summary

- An LLM call is an HTTP POST. The response tells you why it stopped through
  `finish_reason`.
- An agent is a loop that branches on `finish_reason`: run tools on `tool_calls`, answer
  on `stop`. You can write it in about 20 lines.
- A framework like Strands runs that loop and builds tool schemas from your functions,
  so you write tools, not plumbing.

## Next

**Module 2: Real data with MCP.** The agent still guesses about your Akamai cloud account. Next you
will connect a read-only MCP server so it reads your actual Akamai Cloud resources instead of making them up.